# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 clinical colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata as a single object
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

print("\nDataset citation:")
print(metadata.citeAs)

# Show dataset keywords
print("\nDataset keywords:")
pprint.pprint(getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Every entity is referenced by its `@id`. We will enumerate all record sets, their associated fields, and columns, using their `@id`.

In [ ]:
# Enumerate record sets and their fields

# The dataset may have multiple record sets; get them from metadata.recordSet
record_set_objs = getattr(metadata, 'recordSet', [])
if not record_set_objs:
    print("No record sets defined directly in metadata. Attempting to use mlcroissant Dataset API.")
    # Use Dataset API to enumerate record sets
    record_sets_ids = dataset.record_sets.keys() if hasattr(dataset, 'record_sets') else []
else:
    # If metadata.recordSet exists, extract @id
    record_sets_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_set_objs]

# If above is empty, fallback to API
if not record_sets_ids:
    record_sets_ids = list(dataset.record_sets.keys())

print("Record sets found:")
for rs_id in record_sets_ids:
    print(f"- Record set @id: {rs_id}")
    # For each record set, enumerate its fields
    rs_obj = dataset.record_sets.get(rs_id) if hasattr(dataset, 'record_sets') else None
    if rs_obj and getattr(rs_obj, 'fields', None):
        print("  Fields:")
        for field in rs_obj.fields:
            fid = getattr(field, '@id', None)
            fname = getattr(field, 'name', None)
            print(f"    - Field @id: {fid}, name: {fname}")
            # Show columns if possible
            col = getattr(field, 'column', None)
            if col:
                if isinstance(col, list):
                    for c in col:
                        cid = c.get('@id') if isinstance(c, dict) else c
                        print(f"      - Column @id: {cid}")
                else:
                    cid = col.get('@id') if isinstance(col, dict) else col
                    print(f"      - Column @id: {cid}")
    else:
        print("  No fields found or unable to enumerate fields.")

# As example, load a few sample records from a record set
print("\nSample records from each record set:")
for rs_id in record_sets_ids[:1]:  # Show sample for first record set
    for rec in dataset.records(record_set=rs_id):
        print(rec)
        break  # Only first record

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using their @id
record_sets = record_sets_ids
dataframes = {}

for rs_id in record_sets:
    print(f"\nLoading records for record set @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns in record set {rs_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data by attributes.

- Remove outliers
- Normalize numeric data
- Group data by a key attribute

All columns and fields are referenced by their `@id`.

In [ ]:
# Pick a record set to analyze
if dataframes:
    # Use the first available record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using record set @id: {record_set_id}")
    # Try to identify numeric fields (for demo, look for fields like 'age' or 'interval_months')
    numeric_col_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'months' in col.lower() or 'year' in col.lower()]
    if numeric_col_candidates:
        numeric_field_id = numeric_col_candidates[0]  # Use first candidate
        print(f"Numeric field selected: {numeric_field_id}")
        threshold = 40
        # Filter records with age > threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by another field, e.g. 'sex' or 'anatomical_location'
        group_field_candidates = [col for col in df.columns if 'sex' in col.lower() or 'anatomical' in col.lower() or 'site' in col.lower()]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable field for grouping found.")
    else:
        print("No numeric fields found in the DataFrame.")
else:
    print("No DataFrames extracted from any record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

All columns referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only visualize if numeric fields exist
if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(6,3))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if 'group_field_id' in locals():
        plt.figure(figsize=(7,4))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Explored dataset metadata using `mlcroissant`.
- Enumerated all record sets via their `@id`.
- Loaded record set data and referenced fields/columns by their `@id`.
- Performed EDA including filtering, normalization, and grouping by key attributes.
- Visualized numeric field distributions and relations.

**The FAIR^2 dataset provides a valuable resource for clinicopathological research in colorectal cancer survivors, with detailed annotated records, robust preprocessing, and clear use cases for biomarker analysis and anatomical stratification.**